Chapter 8: Web Scraping

Context:
Data science needs data, and a lot of useful data lives on websites rather than in ready-made files. Web scraping means writing code that downloads web pages and pulls information out of them automatically.

Problem:
Collect specific information from web pages, such as email addresses, phone numbers, links and table data, without copying it by hand.

Solution:
Download a page's HTML code with requests. Find information first with simple text searches (find), then with regular expressions, which are flexible patterns that match text such as email addresses. Finally, use Beautiful Soup to pull data straight out of HTML elements (links, labels and tables), and save the results in a pandas dataframe.

Tools:
requests, re (regular expressions), pandas, Beautiful Soup (bs4) with the lxml parser

Implementation:

In [15]:
import requests
urltoget = 'https://bradfordtuckfield.com/indexarchive20210903.html'
pagecode = requests.get(urltoget)
print(pagecode.text[0:600])

<?xml version="1.0" encoding="utf-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en-US" lang="en-US">
  <head><meta http-equiv="Content-Type" content="text/html; charset=utf-8">

    <title>Bradford Tuckfield</title>
    <meta name="description" content="Bradford Tuckfield" />
    <meta name="keywords" content="Bradford Tuckfield" />
    <meta name="google-site-verification" content="eNw-LEFxVf71e-ZlYnv5tGSxTZ7V32coMCV9bxS3MGY" />
<link rel="stylesheet" type="text/css" href=


Scraping an email address with find(): locate the text "Email:" in the page code, then print the characters around it.

In [16]:
urltoget = 'https://bradfordtuckfield.com/contactscrape.html'
pagecode = requests.get(urltoget)
mail_beginning=pagecode.text.find('Email:')
print(mail_beginning)
print(pagecode.text[(mail_beginning):(mail_beginning+80)])


print(pagecode.text[(mail_beginning+38):(mail_beginning+64)])

511
Email:  <label class="email" href="#">demo@bradfordtuckfield.com</label>
</div>
demo@bradfordtuckfield.com


In [17]:
urltoget = 'https://bradfordtuckfield.com/contactscrape.html'
pagecode = requests.get(urltoget)
at_beginning=pagecode.text.find('@')
print(at_beginning)

print(pagecode.text[(at_beginning-4):(at_beginning+22)])

553
demo@bradfordtuckfield.com


Problem: find() only works if we know exactly how many characters to skip. Regular expressions solve this by searching for patterns instead of exact text.

In [18]:
import re
print(re.search(r'recommend','irrelevant text I recommend irrelevant text').span())
print(re.search('rec+om+end', 'irrelevant text I recommend irrelevant text').span())

(18, 27)
(18, 27)


In [19]:

print(re.search('rec+om+end','irrelevant text I recomend irrelevant text').span())
print(re.search('rec+om+end','irrelevant text I reccommend irrelevant text').span())
print(re.search('rec*om+end','irrelevant text I reommend irrelevant text').span())
print(re.search('rec+om+en[dt]','irrelevant text I recomment irrelevant text').span())

(18, 26)
(18, 28)
(18, 26)
(18, 27)


In [20]:

print(re.search('10*','My bank balance is 100').span())
print(re.search('10*','My bank balance is 1').span())
print(re.search('10*','My bank balance is 1000').span())
print(re.search('\d0*','My bank balance is 9000').span())
print(re.search('10*','My bank balance is 1000000').span())

(19, 22)
(19, 20)
(19, 23)
(19, 23)
(19, 26)


<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_637/4114001433.py:4: SyntaxWarning: invalid escape sequence '\d'
  print(re.search('\d0*','My bank balance is 9000').span())


In [21]:

print(re.search('Clarke?','Please refer questions to Mr. Clark').span())
print(re.search('99\+12=111','Example addition: 99+12=111').span())
print(re.search('Clarke\?','Is anyone here named Clarke?').span())
print(re.search('Clarke?\?','Is anyone here named Clark?').span())
print(re.search(r'\\',r'The escape character is \\').span())
print(re.search('\d','The loneliest number is 1').span())

(30, 35)
(18, 27)
(21, 28)
(21, 27)
(24, 25)
(24, 25)


<>:2: SyntaxWarning: invalid escape sequence '\+'
<>:3: SyntaxWarning: invalid escape sequence '\?'
<>:4: SyntaxWarning: invalid escape sequence '\?'
<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\+'
<>:3: SyntaxWarning: invalid escape sequence '\?'
<>:4: SyntaxWarning: invalid escape sequence '\?'
<>:6: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_637/192163068.py:2: SyntaxWarning: invalid escape sequence '\+'
  print(re.search('99\+12=111','Example addition: 99+12=111').span())
/tmp/ipykernel_637/192163068.py:3: SyntaxWarning: invalid escape sequence '\?'
  print(re.search('Clarke\?','Is anyone here named Clarke?').span())
/tmp/ipykernel_637/192163068.py:4: SyntaxWarning: invalid escape sequence '\?'
  print(re.search('Clarke?\?','Is anyone here named Clark?').span())
/tmp/ipykernel_637/192163068.py:6: SyntaxWarning: invalid escape sequence '\d'
  print(re.search('\d','The loneliest number is 1').span())


In [22]:

print(re.search('[a-z]','My Twitter is @fake; my email is abc@def.com').span())
print(re.search('[A-Z]','My Twitter is @fake; my email is abc@def.com').span())
print(re.search('Manchac[a|k]','Lets drive on Manchaca.').span())

(1, 2)
(0, 1)
(14, 22)


In [23]:

print(re.search('school.*\.pdf$','schoolforgottenname.pdf').span())
print(re.search('school.*\.pdf$','school.pdf').span())
print(re.search('school.*\.pdf$','schoolothername.pdf').span())
# print(re.search('school.*\.pdf$','othername.pdf').span())               # error: doesn't start with school
# print(re.search('school.*\.pdf$','schoolothernamepdf').span())          # error: no period before pdf
# print(re.search('school.*\.pdf$','schoolforgottenname.pdf.exe').span()) # error: doesn't end with .pdf

(0, 23)
(0, 10)
(0, 19)


<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_637/790333795.py:1: SyntaxWarning: invalid escape sequence '\.'
  print(re.search('school.*\.pdf$','schoolforgottenname.pdf').span())
/tmp/ipykernel_637/790333795.py:2: SyntaxWarning: invalid escape sequence '\.'
  print(re.search('school.*\.pdf$','school.pdf').span())
/tmp/ipykernel_637/790333795.py:3: SyntaxWarning: invalid escape sequence '\.'
  print(re.search('school.*\.pdf$','schoolothername.pdf').span())


Using a regular expression to find email addresses: <letters> @ <letters> . <letters>

In [24]:
print(re.search('[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+', 'My Twitter is @fake; my email is abc@def.com').span())

(33, 44)


<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_637/1720429094.py:1: SyntaxWarning: invalid escape sequence '\.'
  print(re.search('[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+', 'My Twitter is @fake; my email is abc@def.com').span())


In [25]:
import requests
urltoget = 'https://bradfordtuckfield.com/contactscrape2.html'
pagecode = requests.get(urltoget)
print(pagecode.text)

allmatches=re.finditer('[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+',pagecode.text)

alladdresses = []
for match in allmatches:
    alladdresses.append(match[0])
print(alladdresses)

<>:6: SyntaxWarning: invalid escape sequence '\.'
<>:6: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_637/17159662.py:6: SyntaxWarning: invalid escape sequence '\.'
  allmatches=re.finditer('[a-zA-Z]+@[a-zA-Z]+\.[a-zA-Z]+',pagecode.text)


<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>contact scraping</title>
</head>
<body>
    
<h2>Demo website for contact scraping</h2>

<p>
Here is some irrelevant text. A contact email is abc@abc.com. Another email is def@def.com. You can also try to email ghi@ghi.com. There's also jkl@jkl.org, and also mno@mno.net. Here is some more irrelevant text. Another email address is pqr@pqr.edu. More irrelevant text. stu@stu.com is another example email address.
</p>

    
</body>
</html>

['abc@abc.com', 'def@def.com', 'ghi@ghi.com', 'jkl@jkl.org', 'mno@mno.net', 'pqr@pqr.edu', 'stu@stu.com']


In [26]:
import pandas as pd
alladdpd=pd.DataFrame(alladdresses)
print(alladdpd)

alladdpd=alladdpd.sort_values(0,ascending=False)
alladdpd.to_csv('alladdpd20220720.csv')
print(alladdpd)

             0
0  abc@abc.com
1  def@def.com
2  ghi@ghi.com
3  jkl@jkl.org
4  mno@mno.net
5  pqr@pqr.edu
6  stu@stu.com
             0
6  stu@stu.com
5  pqr@pqr.edu
4  mno@mno.net
3  jkl@jkl.org
2  ghi@ghi.com
1  def@def.com
0  abc@abc.com


Beautiful Soup: get data straight from HTML elements without writing regular expressions.

In [27]:
import requests
from bs4 import BeautifulSoup
URL = 'https://bradfordtuckfield.com/indexarchive20210903.html'
response = requests.get(URL)
soup = BeautifulSoup(response.text, 'lxml')
all_urls = soup.find_all('a')
for each in all_urls:
  print(each['href'])

https://nostarch.com/Dive-Into-Algorithms
https://www.penguinrandomhouse.com/books/645953/dive-into-algorithms-by-bradford-tuckfield/9781718500686/
https://www.amazon.com/dp/1718500688
https://www.amazon.com/Applied-Unsupervised-Learning-relationships-hierarchical/dp/1789956390/
final20190428.pdf
http://thedreamtigers.com/
https://kmbara.com
https://www.theamericanconservative.com/author/bradford-tuckfield/
http://www.the-american-interest.com/byline/bradford-tuckfield/
tai.pdf
tai2018.pdf
https://www.nationalaffairs.com/authors/detail/bradford-tuckfield
nationalaffairs.pdf
https://americanaffairsjournal.org/2017/10/the-incoherence-of-the-economists/
borges.pdf
https://quillette.com/author/bradford-tuckfield/
https://vitadabrutto.wordpress.com/2019/03/13/disuguaglianze-estetiche-ed-economia-del-sesso/
https://nas.org/blogs/dicta/avoiding_scholarships_dead_ends
http://www.newenglishreview.org/custpage.cfm/frm/167043/sec_id/167043
https://web.archive.org/web/20170705161916/http://www.tas

/tmp/ipykernel_637/2539103659.py:5: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(response.text, 'lxml')


In [28]:
URL = 'https://bradfordtuckfield.com/contactscrape.html'
response = requests.get(URL)
soup = BeautifulSoup(response.text, 'lxml')
email = soup.find('label',{'class':'email'}).text
mobile = soup.find('label',{'class':'mobile'}).text
website = soup.find('a',{'class':'website'}).text
print("Email : {}".format(email))
print("Mobile : {}".format(mobile))
print("Website : {}".format(website))

Email : demo@bradfordtuckfield.com
Mobile : +1 879-890-9767
Website : www.bradfordtuckfield.com


In [29]:
URL = 'https://bradfordtuckfield.com/user_detailsscrape.html'
response = requests.get(URL)
soup = BeautifulSoup(response.text, 'lxml')
all_user_entries = soup.find_all('tr',{'class':'user-details'})
for each_user in all_user_entries:
   user = each_user.find_all("td")
   print("User Firstname : {}, Lastname : {}, Age: {}".format(user[0].text, user[1].text, user[2].text))


User Firstname : Jill, Lastname : Smith, Age: 50
User Firstname : Eve, Lastname : Jackson, Age: 44
User Firstname : John, Lastname : Jackson, Age: 24
User Firstname : Kevin, Lastname : Snow, Age: 34


Conclusion:
Web scraping turns web pages into data. We downloaded HTML with requests and found an email address with find(), but that needed exact character counts. Regular expressions (+, *, ?, \, [ ], |, ., $) let us search for flexible patterns, so we could collect every email address on a page into a pandas dataframe and save it as a .csv file. Beautiful Soup made it even easier by reading HTML elements directly: links, labeled contact details and table rows. Advanced scraping covers dynamic pages (Selenium), APIs and slowing down requests, and scraping should always be done legally and ethically.